# 1. Train/Test Split

Before estimating the Bayesian Marketing Mix Model, the dataset is divided into training and testing subsets. Since Marketing Mix Models operate on time-series data, a chronological split is applied instead of random sampling in order to preserve the temporal structure of the observations.

The first 80% of weekly observations are used for model estimation, while the remaining 20% are reserved for out-of-sample evaluation. This setup allows the model to be evaluated on unseen future periods and reflects realistic forecasting conditions in marketing analytics.

The dataset consists of weekly revenue observations together with multiple paid media channels and control variables. Revenue is treated as the dependent variable, while media spend variables are used as the main explanatory variables in the Bayesian MMM specification.


In [ ]:
# Core libraries
import os
import requests
import pyreadr
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Create the same folder names used in VSCode
os.makedirs("raw_data", exist_ok=True)
os.makedirs("processed_data", exist_ok=True)

# Download Robyn RData file directly from GitHub
url = "https://github.com/facebookexperimental/Robyn/raw/main/R/data/dt_simulated_weekly.RData"

response = requests.get(url)

with open("raw_data/dt_simulated_weekly.RData", "wb") as f:
    f.write(response.content)

# Read RData file
result = pyreadr.read_r("raw_data/dt_simulated_weekly.RData")

# Extract dataset using the same object name as in VSCode
df = result["dt_simulated_weekly"]

# Save a CSV copy using the same file name as in VSCode
df.to_csv("processed_data/dt_simulated_weekly.csv", index=False)

# Load processed dataset
df = pd.read_csv("processed_data/dt_simulated_weekly.csv")

# Convert date column
df["DATE"] = pd.to_datetime(df["DATE"])

# Sort chronologically
df = df.sort_values("DATE").reset_index(drop=True)

# Define variables
target = "revenue"

media_cols = [
    "tv_S",
    "ooh_S",
    "print_S",
    "facebook_S",
    "search_S"
]

control_cols = [
    "competitor_sales_B",
    "newsletter"
]

# Create chronological train-test split
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

# Check split sizes
print("Total observations:", len(df))
print("Train observations:", len(train_df))
print("Test observations:", len(test_df))

# Preview
df.head()


In [ ]:
# Visualize chronological train-test split
plt.figure(figsize=(15, 5))

plt.plot(
    train_df["DATE"],
    train_df[target],
    label="Train"
)

plt.plot(
    test_df["DATE"],
    test_df[target],
    label="Test"
)

plt.title("Chronological Train-Test Split")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()

plt.show()


# Findings

The chronological train-test split preserves the temporal structure of the dataset and avoids information leakage from future observations into the training process. The testing period represents the most recent weeks in the dataset and therefore provides a realistic out-of-sample evaluation setting for the Bayesian Marketing Mix Model.

The visualization shows that the test set maintains similar seasonal patterns and volatility characteristics observed in the training period. This suggests that the holdout data remains representative of the underlying revenue-generating process while still containing unseen future observations for model evaluation.


# 2. Scaling

Bayesian Marketing Mix Models are sensitive to differences in variable magnitudes because Markov Chain Monte Carlo (MCMC) sampling may become unstable when predictors operate on vastly different scales.

To improve numerical stability and sampling efficiency, media spend variables and the target variable are scaled using MaxAbsScaler. This transformation preserves the relative structure of the data while normalizing the magnitude of the variables.


In [ ]:
# Scaling
from sklearn.preprocessing import MaxAbsScaler

# Initialize scalers
media_scaler = MaxAbsScaler()
target_scaler = MaxAbsScaler()


In [ ]:
# Scaling and model-ready dataset preparation


# Copy train and test sets
train_model_df = train_df.copy()
test_model_df = test_df.copy()

# Newsletter has no variation in the training period, so it is excluded
control_cols = [
    "competitor_sales_B"
]

# Scale media channels for diagnostic consistency
media_scaler = MaxAbsScaler()

train_model_df[media_cols] = media_scaler.fit_transform(
    train_model_df[media_cols]
)

test_model_df[media_cols] = media_scaler.transform(
    test_model_df[media_cols]
)

# Standardize continuous control variable
control_mean = train_model_df["competitor_sales_B"].mean()
control_std = train_model_df["competitor_sales_B"].std()

train_model_df["competitor_sales_B"] = (
    train_model_df["competitor_sales_B"] - control_mean
) / control_std

test_model_df["competitor_sales_B"] = (
    test_model_df["competitor_sales_B"] - control_mean
) / control_std

# Prepare feature matrices for MMM
X_train = train_model_df[
    ["DATE"] + media_cols + control_cols
].copy()

X_test = test_model_df[
    ["DATE"] + media_cols + control_cols
].copy()

# Prepare target
y_train = train_model_df[target].copy()
y_train.name = "y"

y_test = test_model_df[target].copy()
y_test.name = "y"

# Check prepared training data
X_train.describe()


In [ ]:
# Check scaled values
train_df[media_cols + [target]].describe()


# Findings

The scaling procedure successfully normalized the media spend variables and the target variable into a comparable numerical range. This transformation improves numerical stability during Bayesian inference and facilitates more efficient MCMC sampling.

The descriptive statistics also reveal that several media channels are sparse, with median values close to zero. This indicates that advertising activity is concentrated in specific periods rather than being continuously active across all weeks, which is consistent with realistic campaign-based marketing behavior.


# 3. PyMC-Marketing Setup

To estimate the Bayesian Marketing Mix Model, the PyMC-Marketing framework is used. PyMC-Marketing is a probabilistic marketing analytics library built on top of PyMC and provides specialized components for Bayesian MMM estimation, including adstock and saturation transformations.

Using PyMC-Marketing enables explicit specification of structural assumptions such as carryover effects and diminishing returns, which are central to the research question of this seminar paper. In particular, the Bayesian MMM baseline incorporates manually specified structural priors that will later be compared against the implicit learned priors of PFN-based models.


In [ ]:
# Test PyMC-Marketing installation
import pymc_marketing

print(pymc_marketing.__version__)


In [ ]:
pip install pymc-marketing


In [ ]:
import pymc_marketing

print(pymc_marketing.__version__)


# Findings

PyMC-Marketing was successfully installed and imported. The installed version is 0.19.4, which provides the required MMM components for Bayesian media mix modeling.

The environment returned a compiler-related warning indicating that PyTensor may not use C-compiled implementations. This does not prevent model estimation, but it may reduce computational speed during sampling.


# 4. Bayesian Model Specification

The Bayesian Marketing Mix Model is specified using explicit structural assumptions regarding advertising carryover effects and diminishing returns.

To capture delayed advertising effects over time, a geometric adstock transformation is applied to the media channels. This transformation models the persistence of advertising impact across multiple weeks and reflects the temporal carryover behavior identified during the exploratory analysis.

To account for nonlinear media effectiveness and diminishing marginal returns, a Hill saturation transformation is incorporated. This allows the model to represent the realistic phenomenon where incremental revenue gains weaken at higher advertising spend levels.

In addition to paid media channels, control variables are included in the specification to reduce omitted variable bias and improve causal identification.


In [ ]:
# PyMC-Marketing MMM components
from pymc_marketing.mmm.multidimensional import MMM
from pymc_marketing.mmm import GeometricAdstock
from pymc_marketing.mmm import LogisticSaturation

# Define Bayesian MMM
mmm = MMM(
    date_column="DATE",
    channel_columns=media_cols,
    control_columns=control_cols,
    adstock=GeometricAdstock(l_max=8),
    saturation=LogisticSaturation(),
    yearly_seasonality=2,
)


In [ ]:
# Define variables
target = "revenue"

# Media channels
media_cols = [
    "tv_S",
    "ooh_S",
    "print_S",
    "facebook_S",
    "search_S"
]

# Control variables
control_cols = [
    "competitor_sales_B",
    "newsletter"
]


In [ ]:
# Define Bayesian MMM
mmm = MMM(
    date_column="DATE",
    channel_columns=media_cols,
    control_columns=control_cols,
    adstock=GeometricAdstock(l_max=8),
    saturation=HillSaturation(),
)


In [ ]:
# Inspect default priors
mmm.default_model_config


# Findings

The Bayesian MMM was successfully specified using explicit structural transformations for both carryover effects and diminishing returns.

A geometric adstock transformation was selected to model the delayed persistence of advertising effects across multiple weeks, consistent with the lagged correlation patterns observed during exploratory analysis.

In addition, a Hill saturation transformation was incorporated to capture nonlinear advertising effectiveness and diminishing marginal returns at higher spend levels.

The resulting specification represents a structurally informed Bayesian benchmark model that can later be compared against PFN-based models relying on implicit learned priors rather than manually defined marketing assumptions.


# 5. Priors

Bayesian Marketing Mix Models require prior distributions to encode assumptions regarding parameter magnitudes and structural behavior before observing the data. Priors play a particularly important role in MMM settings because media effects are often noisy, nonlinear, and partially identifiable.

In this specification, weakly informative priors are used in order to regularize parameter estimation while still allowing the observed data to drive posterior inference. The use of explicit priors represents a key conceptual difference between Bayesian MMMs and PFN-based models, which instead rely on implicit priors learned during pretraining on synthetic data-generating processes.


In [ ]:
# Display model configuration
mmm


In [ ]:
# Inspect default model configuration and priors
mmm.default_model_config


# Findings

The Bayesian MMM uses weakly informative default priors that regularize the estimation of media, control, adstock, saturation, and likelihood parameters.

The adstock parameter is assigned a Beta(1, 3) prior, which favors moderate carryover effects while still allowing channel-specific variation. This is consistent with the exploratory evidence showing persistent but decaying lagged correlations.

The saturation parameters use HalfNormal priors, ensuring positive channel effects and allowing the model to capture nonlinear diminishing returns. This directly supports the model's role as a structurally informed Bayesian benchmark.

Compared to PFN-based models, these priors are explicit and interpretable, which makes the Bayesian MMM suitable as the main reference model for evaluating attribution and ROAS estimates.


# 6. Model Fitting

The Bayesian Marketing Mix Model is estimated using Markov Chain Monte Carlo (MCMC) sampling. Bayesian inference allows the model to estimate posterior distributions for all parameters instead of relying on single-point estimates.

This probabilistic framework is particularly valuable in MMM settings because media attribution is inherently uncertain and often affected by temporal dependencies, nonlinearities, and partial identifiability.

The model is fitted on the training dataset only, while the holdout period remains unseen for later out-of-sample evaluation.


In [ ]:
# Recreate train-test split from unscaled data
df_model = df.copy()

split_idx = int(len(df_model) * 0.8)

train_model_df = df_model.iloc[:split_idx].copy()
test_model_df = df_model.iloc[split_idx:].copy()


In [ ]:
# Prepare features and target
X_train = train_model_df[
    ["DATE"] + media_cols + control_cols
].copy()

# Add small epsilon to avoid exact zero media values
epsilon = 1e-6
X_train[media_cols] = X_train[media_cols] + epsilon

# Target separately
y_train = train_model_df[target].copy()
y_train.name = "y"


In [ ]:
# Import required classes
import numpy as np

from pymc_marketing.prior import Prior
from pymc_marketing.mmm.multidimensional import MMM
from pymc_marketing.mmm.components.adstock import GeometricAdstock
from pymc_marketing.mmm.components.saturation import LogisticSaturation


In [ ]:
# Define safer prior configuration
model_config = {
    "intercept": Prior("Normal", mu=0, sigma=2),

    "likelihood": Prior(
        "Normal",
        sigma=Prior("HalfNormal", sigma=2),
        dims="date"
    ),

    "gamma_control": Prior("Normal", mu=0, sigma=2, dims="control"),
    "gamma_fourier": Prior("Laplace", mu=0, b=1, dims="fourier_mode"),

    "adstock_alpha": Prior(
        "Beta",
        alpha=2,
        beta=2,
        dims="channel"
    ),

    "saturation_beta": Prior(
        "HalfNormal",
        sigma=1.5,
        dims="channel"
    ),
}


In [ ]:
# Define Bayesian MMM
mmm = MMM(
    date_column="DATE",
    channel_columns=media_cols,
    control_columns=control_cols,
    target_column="y",
    adstock=GeometricAdstock(l_max=8),
    saturation=LogisticSaturation(),
    model_config=model_config,
)


In [ ]:
# Fit Bayesian MMM

idata = mmm.fit(
    X=X_train,
    y=y_train,
    chains=4,
    cores=2,
    draws=1000,
    tune=2000,
    target_accept=0.95,
    random_seed=42,
)


In [ ]:
summary_diag = az.summary(
    idata,
    var_names=[
        "adstock_alpha",
        "saturation_beta",
        "gamma_control"
    ],
    round_to=3
)

summary_diag


In [ ]:
list(idata.posterior.data_vars)


In [ ]:
idata.sample_stats


In [ ]:
print("Divergences:", int(idata.sample_stats["diverging"].sum()))
print("Max tree depth reached:", int(idata.sample_stats["reached_max_treedepth"].sum()))
